In [0]:
%pip install pytest

In [0]:
import sys, os, uuid
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0

In [0]:
from etl.migrate import run_migrations
run_migrations(spark, f"{repo}/sql/ddl")

In [0]:
from etl.registry import register_files
register_files(spark)
display(spark.sql("SELECT file_id, raw_table, part_no, status, size_bytes, file_path FROM dld.meta.file_registry ORDER BY file_id"))

In [0]:
from etl.registry import TABLES
from etl.load_csv import load_csv
run_id = str(uuid.uuid4())
for t, (_, _, fmt) in TABLES.items():
    if fmt == "csv":
        load_csv(spark, t, run_id)

In [0]:
display(spark.sql("SELECT target_table, rows_loaded, file_path FROM dld.meta.load_log ORDER BY target_table"))
for t, (_, _, fmt) in TABLES.items():
    if fmt == "csv":
        r = spark.sql(f"SELECT count(*) n, sum(CASE WHEN _row_num IS NULL OR _file_id IS NULL THEN 1 ELSE 0 END) bad FROM dld.raw.{t}").first()
        print(t.ljust(20), r.n, "rows,", r.bad, "missing lineage")

In [0]:
p = spark.table("dld.meta.file_registry").where("raw_table = 'e2_lkp_areas'").first().file_path
x = spark.read.format("excel").option("headerRows", 1).load(p)
x.printSchema()
display(x.limit(5))